# Day 2 - Từ annotation YOLO đến detector YOLO26

Notebook này chạy **sau** khi nhãn 50 ảnh train đã qua cổng gold-set. Tập `eval` 10 ảnh không được dùng để train.

Bài cá nhân. Hạn nộp: 11:59 trưa ngày hôm sau buổi lab (UTC+7); ca 09/11/2026 nộp trước 11:59 ngày 10/11/2026.

## Kiểm tra trước khi chạy

- Upload bản sao `Day2-Lab/` đã có `dataset/labels/train/` của bạn.
- Không upload `gold_train_labels/`.
- Chạy từ trên xuống dưới; giữ nguyên `data.yaml` và split eval.
- mAP đo model; F1/IoU ở script trước đó đo chất lượng nhãn con người.

In [ ]:
%pip -q install -U ultralytics

from pathlib import Path
import json
import shutil
import sys
import torch
import yaml
from ultralytics import YOLO

# Nếu upload vào /content/Day2-Lab, giữ giá trị này. Nếu mount Drive, sửa một dòng.
LAB_ROOT = Path('/content/Day2-Lab')
if not LAB_ROOT.exists():
    LAB_ROOT = Path('Day2-Lab')
DATA_YAML = LAB_ROOT / 'data.yaml'
TRAIN_IMAGES = LAB_ROOT / 'dataset/images/train'
TRAIN_LABELS = LAB_ROOT / 'dataset/labels/train'
EVAL_IMAGES = LAB_ROOT / 'dataset/images/eval'
EVAL_LABELS = LAB_ROOT / 'dataset/labels/eval'
RUNS = LAB_ROOT / 'outputs/yolo26_runs'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# Ultralytics resolves a relative `path:` in data.yaml against its own global
# datasets dir, not against data.yaml's own folder. Rewrite it to an absolute
# path so training works regardless of where Day2-Lab/ was uploaded.
_data_cfg = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
_data_cfg['path'] = str((LAB_ROOT / 'dataset').resolve())
DATA_YAML.write_text(yaml.dump(_data_cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')

print('LAB_ROOT:', LAB_ROOT.resolve())
print('device:', DEVICE)

In [ ]:
def image_stems(directory):
    return {p.stem for ext in ('*.jpg', '*.jpeg', '*.png') for p in directory.glob(ext)}

train_images = image_stems(TRAIN_IMAGES)
train_labels = {p.stem for p in TRAIN_LABELS.glob('*.txt')}
eval_images = image_stems(EVAL_IMAGES)
eval_labels = {p.stem for p in EVAL_LABELS.glob('*.txt')}
assert len(train_images) == 50, f' cần 50 train images, thấy {len(train_images)}'
assert train_images == train_labels, f'train images/labels lệch: {sorted(train_images ^ train_labels)[:5]}'
assert len(eval_images) == 10 and eval_images == eval_labels, 'eval phải đủ 10 ảnh và nhãn'
print(f'OK: {len(train_images)} train + {len(eval_images)} eval; eval không đi vào train.')

## Huấn luyện

Bài thực hành dùng checkpoint `yolo26n.pt`. Với 50 ảnh, dùng model nano và ít epoch để minh họa vòng lặp dữ liệu; thời gian/metric không dùng làm benchmark sản phẩm.

In [ ]:
model = YOLO('yolo26n.pt')
train_result = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=-1,
    device=DEVICE,
    project=str(RUNS),
    name='vehicle_annotation',
    exist_ok=True,
    seed=20260909,
    plots=True,
)
BEST = Path(model.trainer.best)
print('best checkpoint:', BEST)

## Đánh giá đúng tập eval

`data.yaml` trỏ `val` tới 10 ảnh eval đã có nhãn. Không đổi `val` sang train để làm đẹp mAP. Ghi lại mAP50, mAP50-95, precision và recall.

Tập `eval` cũng được dùng làm validation trong quá trình train và chọn `best.pt`, nên không phải test set độc lập.

In [ ]:
best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(DATA_YAML), split='val', device=DEVICE, plots=True, project=str(RUNS), name='vehicle_eval', exist_ok=True)
summary = {
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'map50': float(metrics.box.map50),
    'map50_95': float(metrics.box.map),
    'eval_images': len(eval_images),
    'checkpoint': str(BEST),
}
metrics_path = LAB_ROOT / 'outputs/model_eval_metrics.json'
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))

In [ ]:
# Kiểm tra trực quan một vài ảnh eval; đây là bước bắt buộc trong báo cáo.
prediction_dir = LAB_ROOT / 'outputs/eval_predictions'
results = best_model.predict(source=str(EVAL_IMAGES), conf=0.25, device=DEVICE, save=True, project=str(prediction_dir), name='visuals', exist_ok=True)
PRED_SAVE_DIR = Path(results[0].save_dir)
print('predictions saved to:', PRED_SAVE_DIR)
archive = shutil.make_archive(str(LAB_ROOT / 'outputs/day2_colab_outputs'), 'zip', LAB_ROOT / 'outputs')
print('Đã tạo:', archive)

## Xem trực tiếp kết quả dự đoán

Hiển thị 10 ảnh eval kèm box dự đoán ngay trong notebook, không cần tải file `.zip` về mới xem được.

In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image

pred_images = sorted(PRED_SAVE_DIR.glob('*.jpg'))
cols = 5
rows = math.ceil(len(pred_images) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
for ax, img_path in zip(axes.flat, pred_images):
    ax.imshow(Image.open(img_path))
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(pred_images):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Câu hỏi phân tích để nộp

1. Lớp nào có recall thấp nhất? Chỉ ra một ảnh eval minh họa.
2. Nếu mAP50 cao nhưng mAP50-95 thấp, box đang gặp vấn đề gì?
3. Có lỗi `truck`/`bus`/`van`/`car` nào liên quan tới guideline không?
4. Vì sao không dùng 10 ảnh eval để train dù chúng có nhãn?